# Отчёт 2. Newton, BFGS и Nelder–Mead

Экспериментальное сравнение методов второго порядка, квазиньютоновской оптимизации и поиска без производных на гладких и мультимодальных функциях.

> **Воспроизводимость.** Для воспроизведения перезапустите ядро и выполните все ячейки по порядку.

[Описание, результаты и инструкция по запуску](../docs/02-classical-methods.md)


# Базовая часть


## Функции и их производные

### Функция Растригина (Rastrigin)

$$
f(x, y) = 20 + x^2 + y^2 - 10 \bigl( \cos(2\pi x) + \cos(2\pi y) \bigr)
$$

Глобальный минимум достигается в точке $(0, 0)$:

$$
f(0, 0) = 20 + 0 + 0 - 10(1 + 1) = 20 - 20 = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
2x + 20\pi \sin(2\pi x) \\[4pt]
2y + 20\pi \sin(2\pi y)
\end{pmatrix}
$$

#### Гессиан

$$
\mathbf{H}_f(x, y) =
\begin{pmatrix}
\dfrac{\partial^2 f}{\partial x^2} & \dfrac{\partial^2 f}{\partial x \partial y} \\[8pt]
\dfrac{\partial^2 f}{\partial y \partial x} & \dfrac{\partial^2 f}{\partial y^2}
\end{pmatrix}
=
\begin{pmatrix}
2 + 40\pi^2 \cos(2\pi x) & 0 \\[4pt]
0 & 2 + 40\pi^2 \cos(2\pi y)
\end{pmatrix}
$$

### Функция "Ящик для яиц" (Eggcrate)

$$
f(x, y) = x^2 + y^2 + 25 \left( \sin^2 x + \sin^2 y \right)
$$

Глобальный минимум достигается в точке $(0, 0)$:

$$
f(0, 0) = 0^2 + 0^2 + 25(\sin^2 0 + \sin^2 0) = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
2x + 25 \sin 2x \\[4pt]
2y + 25 \sin 2y
\end{pmatrix}
$$

#### Гессиан

$$
\mathbf{H}_f(x, y) =
\begin{pmatrix}
\dfrac{\partial^2 f}{\partial x^2} & \dfrac{\partial^2 f}{\partial x \partial y} \\[8pt]
\dfrac{\partial^2 f}{\partial y \partial x} & \dfrac{\partial^2 f}{\partial y^2}
\end{pmatrix}
=
\begin{pmatrix}
2 + 50 \cos 2x & 0 \\[4pt]
0 & 2 + 50 \cos 2y
\end{pmatrix}
$$

### Функция Бута (Booth)

$$
f(x, y) = (x + 2y - 7)^2 + (2x + y - 5)^2
$$

Глобальный минимум достигается в точке $(1, 3)$:

$$
f(1, 3) = (1 + 6 - 7)^2 + (2 + 3 - 5)^2 = 0^2 + 0^2 = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
10x + 8y - 34 \\[4pt]
8x + 10y - 38
\end{pmatrix}
$$

#### Гессиан

$$
\mathbf{H}_f(x, y) =
\begin{pmatrix}
\dfrac{\partial^2 f}{\partial x^2} & \dfrac{\partial^2 f}{\partial x \partial y} \\[8pt]
\dfrac{\partial^2 f}{\partial y \partial x} & \dfrac{\partial^2 f}{\partial y^2}
\end{pmatrix}
=
\begin{pmatrix}
10 & 8 \\
8 & 10
\end{pmatrix}
$$

## Методы базовой части

### Метод Ньютона, Метод BFGS(вызывается из библиотеки), Метод Нелдера-Мида(вызывается из библиотеки)

## Метод Ньютона (собственная реализация)

- Использует вторую производную, посчитанную ранее (матрицу Гессе).
- На каждом шаге решает систему уравнений, чтобы найти направление к минимуму.
- Условие Армихо — пробуем шаг длиной 1, если функция не уменьшается, уменьшаем шаг в 2 раза и повторяем.
- Защита от ошибок: добавляем маленькое число к диагонали гессиана, чтобы он не был вырожденным.
- Если направление получилось "подъёмным" (функция растёт), заменяем его на антиградиент(и получаем градиентный спуск). Можно было без этого, но тогда он часто расходится
- Останавливаемся, когда градиент становится очень маленьким (почти ноль) или шаги слишком короткие.

## Метод BFGS (из библиотеки scipy)

- Требует исключительно градиент, а поведение гессиана приближает на основе прошлых шагов.
- Всегда идёт вниз, потому что его приближение гессиана — положительное.
- Шаг подбирается с помощью условия Вольфе (библиотечная реализация).
- Надёжнее метода Ньютона на сложных функциях, не так сильно зависит от начальной точки.
- Требует больше итераций, чем идеальный Ньютон, зато каждая итерация дешевле.

## Метод Нелдера-Мида (из библиотеки scipy)

- Не использует производные совсем.
- Представляет собой симплекс, который двигается в поисках минимума.
- Не требует вычисления градиентов и гессианов — хорош для страшных функций с громоздкими производными и гессианами.
- Очень медленный, особенно на гладких функциях. Требует много вычислений функции.
- Останавливаемся, когда размер треугольника становится очень маленьким.

## Запуск эксперимента

## Графики

## Код, собранный в одном месте, чтобы запустить

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import pandas as pd
import warnings

def booth(x):
    return (x[0] + 2*x[1] - 7)**2 + (2*x[0] + x[1] - 5)**2
def booth_grad(x):
    return np.array([10*x[0] + 8*x[1] - 34, 8*x[0] + 10*x[1] - 38])
def booth_hess(x):
    return np.array([[10, 8], [8, 10]])

def eggcrate(x):
    return x[0]**2 + x[1]**2 + 25*(np.sin(x[0])**2 + np.sin(x[1])**2)
def eggcrate_grad(x):
    return np.array([2*x[0] + 25*np.sin(2*x[0]), 2*x[1] + 25*np.sin(2*x[1])])
def eggcrate_hess(x):
    return np.array([[2 + 50*np.cos(2*x[0]), 0], [0, 2 + 50*np.cos(2*x[1])]])

def rastrigin(x):
    return 20 + x[0]**2 + x[1]**2 - 10*(np.cos(2*np.pi*x[0]) + np.cos(2*np.pi*x[1]))
def rastrigin_grad(x):
    return np.array([2*x[0] + 20*np.pi*np.sin(2*np.pi*x[0]),
                     2*x[1] + 20*np.pi*np.sin(2*np.pi*x[1])])
def rastrigin_hess(x):
    return np.array([[2 + 40*np.pi**2*np.cos(2*np.pi*x[0]), 0],
                     [0, 2 + 40*np.pi**2*np.cos(2*np.pi*x[1])]])

def armijo(f, x, d, g, c1=1e-4, alpha=1.0, rho=0.5):
    fx = f(x)
    while True:
        if f(x + alpha*d) <= fx + c1*alpha*np.dot(g, d):
            return alpha
        alpha *= rho
        if alpha < 1e-12:
            return 0.0

def newton_method(f, grad, hess, x0, max_iter=1000, tol=1e-6, reg_eps=1e-8):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    values = [f(x)]
    n_iter = 0
    for _ in range(max_iter):
        g = grad(x)
        if np.linalg.norm(g) < tol:
            break
        H = hess(x) + reg_eps * np.eye(len(x))
        d = np.linalg.solve(H, -g)
        if np.dot(g, d) > 0:
            d = -g
        alpha = armijo(f, x, d, g)
        x = x + alpha * d
        history.append(x.copy())
        values.append(f(x))
        n_iter += 1
        if np.linalg.norm(alpha * d) < tol:
            break
    return x, np.array(history), np.array(values), n_iter

def run_bfgs_with_history(f, grad, x0, max_iter=1000, tol=1e-6):
    history = [np.array(x0)]
    values = [f(x0)]
    def callback(xk):
        history.append(np.array(xk))
        values.append(f(xk))
    res = minimize(f, x0, method='BFGS', jac=grad,
                   options={'maxiter': max_iter, 'gtol': tol}, callback=callback)
    return res.x, np.array(history), np.array(values), res.nit, res.success, res.nfev, res.njev

def run_nelder_mead_with_history(f, x0, max_iter=1000, tol=1e-6):
    history = [np.array(x0)]
    values = [f(x0)]
    def callback(xk):
        history.append(np.array(xk))
        values.append(f(xk))
    res = minimize(f, x0, method='Nelder-Mead',
                   options={'maxiter': max_iter, 'xatol': tol}, callback=callback)
    return res.x, np.array(history), np.array(values), res.nit, res.success, res.nfev

def run_experiment(f, grad, hess, x0, name, max_iter=500, tol=1e-6):
    results = {}
    x_opt, hist, vals, nit = newton_method(f, grad, hess, x0, max_iter, tol)
    results['Newton'] = {
        'x_opt': x_opt, 'history': hist, 'values': vals, 'iterations': nit,
        'converged': nit < max_iter, 'f_calls': len(hist), 'grad_calls': nit, 'hess_calls': nit
    }
    x_opt, hist, vals, nit, success, nfev, njev = run_bfgs_with_history(f, grad, x0, max_iter, tol)
    results['BFGS'] = {
        'x_opt': x_opt, 'history': hist, 'values': vals, 'iterations': nit,
        'converged': success, 'f_calls': nfev, 'grad_calls': njev, 'hess_calls': 0
    }
    x_opt, hist, vals, nit, success, nfev = run_nelder_mead_with_history(f, x0, max_iter, tol)
    results['Nelder-Mead'] = {
        'x_opt': x_opt, 'history': hist, 'values': vals, 'iterations': nit,
        'converged': success, 'f_calls': nfev, 'grad_calls': 0, 'hess_calls': 0
    }
    return results

start_points_multi = {
    'Booth': {
        'рядом с глобальным минимумом': [0.0, 0.0],
        'далеко от глобального минимума': [8, 5],
        'очень далеко от глобального минимума': [9, -8]
    },
    'Eggcrate': {
        'рядом с глобальным минимумом': [1, 1],
        'далеко от глобального минимума': [3, 2],
        'очень далеко от глобального минимума': [3.2, 3.2]
    },
    'Rastrigin': {
        'рядом с глобальным минимумом': [0.3, 0.3],
        'далеко от глобального минимума': [1.5, 1.5],
        'очень далеко от глобального минимума': [1.11, 1.11]
    }
}

functions_dict = {
    'Booth': (booth, booth_grad, booth_hess),
    'Eggcrate': (eggcrate, eggcrate_grad, eggcrate_hess),
    'Rastrigin': (rastrigin, rastrigin_grad, rastrigin_hess)
}

all_experiments = {}
all_results_table = []

for name, (f, grad, hess) in functions_dict.items():
    all_experiments[name] = {}
    for case, x0 in start_points_multi[name].items():
        print(f"Запуск {name}, случай: {case}, x0={x0}")
        exp = run_experiment(f, grad, hess, x0, name, max_iter=500, tol=1e-6)
        all_experiments[name][case] = exp
        for method, data in exp.items():
            all_results_table.append({
                'Function': name,
                'Start_case': case,
                'Start_point': str(x0),
                'Method': method,
                'Iterations': data['iterations'],
                'f_calls': data['f_calls'],
                'grad_calls': data['grad_calls'],
                'hess_calls': data['hess_calls'],
                'Converged': data['converged'],
                'Final_f': data['values'][-1]
            })

for name in functions_dict.keys():
    f, _, _ = functions_dict[name]
    if name == 'Booth':
        bounds = [-10, 10, -10, 10]
    elif name == 'Eggcrate':
        bounds = [-4, 4, -4, 4]
    else:
        bounds = [-2, 2, -2, 2]

    x = np.linspace(bounds[0], bounds[1], 500)
    y = np.linspace(bounds[2], bounds[3], 500)
    X, Y = np.meshgrid(x, y)
    Z = np.array([f([xi, yi]) for xi, yi in zip(X.ravel(), Y.ravel())]).reshape(X.shape)

    for case, exp in all_experiments[name].items():
        print(f"\n=== {name} - случай: {case} ===")
        plt.figure(figsize=(12, 4))
        methods = ['Newton', 'BFGS', 'Nelder-Mead']
        for idx, method in enumerate(methods):
            plt.subplot(1, 3, idx+1)
            plt.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.7, linewidths=0.8)
            hist = exp[method]['history']
            if len(hist) > 1:
                plt.plot(hist[:,0], hist[:,1], 'r-', linewidth=1.5, label=method)
            else:
                plt.scatter(hist[0,0], hist[0,1], color='red', s=60, label=method + ' (нет движения)')
            plt.scatter(hist[0,0], hist[0,1], color='blue', s=40, zorder=5, label='start')
            plt.scatter(hist[-1,0], hist[-1,1], color='red', s=40, zorder=5, label='end')
            plt.xlim(bounds[0], bounds[1])
            plt.ylim(bounds[2], bounds[3])
            plt.title(method)
            plt.legend(fontsize='small')
        plt.suptitle(f'{name} - {case}', y=1.02)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 6))
        for method in methods:
            vals = exp[method]['values']
            diff = np.abs(np.array(vals) - 0) + 1e-16
            plt.semilogy(range(len(diff)), diff, marker='.', label=method)
        plt.xlabel('Номер итерации')
        plt.ylabel('|f(x) - f*|')
        plt.title(f'Сходимость методов на {name} (случай: {case})')
        plt.legend()
        plt.grid(True)
        plt.show()

print("\n" + "="*130)
print(f"{'Function':<12} {'Start case':<30} {'Method':<12} {'Iter':<6} {'f_calls':<8} {'grad':<8} {'hess':<8} {'Converged':<8} {'Final f':<12}")
print("="*130)
for row in all_results_table:
    start_case_short = row['Start_case'][:28]
    print(f"{row['Function']:<12} {start_case_short:<30} {row['Method']:<12} {row['Iterations']:<6} {row['f_calls']:<8} {row['grad_calls']:<8} {row['hess_calls']:<8} {row['Converged']:<8} {row['Final_f']:<12.4e}")
print("="*130)

## Вывод для базовой части


### Функция Бута
Метод Ньютона сработал идеально — сошелся за 1 итерацию. Это ожидаемо, так как для квадратичных функций он находит минимум за один шаг.
BFGS тоже справился очень быстро - на каждой точке не более 8 шагов, показав хорошую сходимость, то же самое видно было в первой лабораторной.
Метод Нелдера-Мида шел к минимуму дольше всех - 62-82 итераций, делая много лишних вычислений функции, 119-161.
На этой функции все методы справились, однако метод Нелдера-Мида тут считал функцию много раз.

### Функция "держатель яиц"
Метод Ньютона тут не сошёлся из-за отрицательно определённого гессиана. Поэтому подменяем его на антиградиент и метод ньютона первращается в градиентный спуск. Это спасает метод от расхождения. С такой модификацией он сошёлся быстро, потратил не более 5 шагов.
Метод BFGS как и в прошлой лабораторной сошёлся, довольно быстро.
Метод Нелдера-Мида опять сделал очень много действий, но сошёлся, опять считал функцию много раз.

Можно заметить, что в 2 из 3 случаев все методы радостно улетели в локальный минимум, когда его увидели, только в случае нахождения в яме глобального минимума, методы сошлись к нему.

### Функция Растригина
Метод Ньютона тут не сошёлся из-за отрицательно определённого гессиана. Поэтому подменяем его на антиградиент и метод ньютона первращается в градиентный спуск. Это спасает метод от расхождения. С такой модификацией он сошёлся быстро, потратил не более 5 шагов.
Метод BFGS как и в прошлой лабораторной сошёлся, довольно быстро.
Метод Нелдера-Мида опять сделал очень много действий, но сошёлся, опять считал функцию много раз.

На этих 2 функциях метод Ньютона разошёлся бы, но модификация, превращающая его в таком случае на градиентный спуск, приводит его к точке минимума, но часто не глобального. 

### Сравнения с первой лабораторной
В целом все методы показали себя одинаково по уровню схождения к локальным/глобальным минимумам, однако найдено 2 примера функций, когда метод Ньютона не сходится. Также метод Нелдера-Мида может съедать очень много ресурсов, если функцию вычислять сложно и делает много итераций.



# __________________________________________________________________________________________________________________________

# Advanced задание

### Функция Вейерштрасса

Для функции Вейерштрасса, заданной конечной суммой (приближение):

$$
f(x, y) = \sum_{k=0}^{20} 0.5^k \cos\Bigl(2\pi \cdot 3^k (x + 0.5)\Bigr) + \sum_{k=0}^{20} 0.5^k \cos\Bigl(2\pi \cdot 3^k (y + 0.5)\Bigr)
$$

Градиент:

$$
\frac{\partial f}{\partial x} = -2\pi \sum_{k=0}^{20} (1.5)^k \sin\Bigl(2\pi \cdot 3^k (x + 0.5)\Bigr)
$$

$$
\frac{\partial f}{\partial y} = -2\pi \sum_{k=0}^{20} (1.5)^k \sin\Bigl(2\pi \cdot 3^k (y + 0.5)\Bigr)
$$

Гессиан:

$$
\frac{\partial^2 f}{\partial x^2} = -4\pi^2 \sum_{k=0}^{20} (4.5)^k \cos\Bigl(2\pi \cdot 3^k (x + 0.5)\Bigr)
$$

$$
\frac{\partial^2 f}{\partial y^2} = -4\pi^2 \sum_{k=0}^{20} (4.5)^k \cos\Bigl(2\pi \cdot 3^k (y + 0.5)\Bigr)
$$

$$
\frac{\partial^2 f}{\partial x \partial y} = 0
$$

Вообще функция Вейерштрасса нигде не дифференцируема, но взятая аппроксимация уже является гладкой.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings

# 1. Функции, их градиенты, гессианы
def booth(x):
    return (x[0] + 2*x[1] - 7)**2 + (2*x[0] + x[1] - 5)**2
def booth_grad(x):
    return np.array([10*x[0] + 8*x[1] - 34, 8*x[0] + 10*x[1] - 38])
def booth_hess(x):
    return np.array([[10, 8], [8, 10]])

def rastrigin(x):
    return 20 + x[0]**2 + x[1]**2 - 10*(np.cos(2*np.pi*x[0]) + np.cos(2*np.pi*x[1]))
def rastrigin_grad(x):
    return np.array([2*x[0] + 20*np.pi*np.sin(2*np.pi*x[0]),
                     2*x[1] + 20*np.pi*np.sin(2*np.pi*x[1])])
def rastrigin_hess(x):
    return np.array([[2 + 40*np.pi**2*np.cos(2*np.pi*x[0]), 0],
                     [0, 2 + 40*np.pi**2*np.cos(2*np.pi*x[1])]])

def weierstrass(x, K=20, a=0.5, b=3):
    xv, yv = x[0], x[1]
    fx, fy = 0.0, 0.0
    for k in range(K+1):
        fx += a**k * np.cos(2 * np.pi * b**k * (xv + 0.5))
        fy += a**k * np.cos(2 * np.pi * b**k * (yv + 0.5))
    return fx + fy

def weierstrass_grad(x, K=20, a=0.5, b=3):
    xv, yv = x[0], x[1]
    gx, gy = 0.0, 0.0
    for k in range(K+1):
        gx += -2 * np.pi * (a * b)**k * np.sin(2 * np.pi * b**k * (xv + 0.5))
        gy += -2 * np.pi * (a * b)**k * np.sin(2 * np.pi * b**k * (yv + 0.5))
    return np.array([gx, gy])

def weierstrass_hess(x, K=20, a=0.5, b=3):
    xv, yv = x[0], x[1]
    hxx, hyy = 0.0, 0.0
    for k in range(K+1):
        hxx += -4 * np.pi**2 * (a * b**2)**k * np.cos(2 * np.pi * b**k * (xv + 0.5))
        hyy += -4 * np.pi**2 * (a * b**2)**k * np.cos(2 * np.pi * b**k * (yv + 0.5))
    return np.array([[hxx, 0.0], [0.0, hyy]])

# 2. Условие Armijo (линейный поиск)
def armijo(f, x, d, g, c1=1e-4, alpha=1.0, rho=0.5, counter=None):
    fx = f(x)
    while True:
        if counter is not None:
            counter['f'] += 1
        if f(x + alpha*d) <= fx + c1*alpha*np.dot(g, d):
            return alpha
        alpha *= rho
        if alpha < 1e-12:
            return 0.0

# 3. Метод Ньютона (без антиградиента)
def newton_method(f, grad, hess, x0, max_iter=1000, tol=1e-6, reg_eps=1e-8):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    values = [f(x)]
    n_iter = 0
    nfev = 1
    njev = 0
    nhev = 0
    for _ in range(max_iter):
        g = grad(x)
        njev += 1
        if np.linalg.norm(g) < tol:
            break
        H = hess(x) + reg_eps * np.eye(len(x))
        nhev += 1
        d = np.linalg.solve(H, -g)
        counter = {'f': 0}
        alpha = armijo(f, x, d, g, counter=counter)
        nfev += counter['f']
        x = x + alpha * d
        nfev += 1
        history.append(x.copy())
        values.append(f(x))
        n_iter += 1
        if np.linalg.norm(alpha * d) < tol:
            break
    return x, np.array(history), np.array(values), n_iter, nfev, njev, nhev

# 4. BFGS
def bfgs_own(f, grad, x0, max_iter=1000, tol=1e-6, c1=1e-4, rho=0.5):
    x = np.array(x0, dtype=float)
    n = len(x)
    B = np.eye(n)
    history = [x.copy()]
    values = [f(x)]
    n_iter = 0
    g = grad(x)
    for _ in range(max_iter):
        if np.linalg.norm(g) < tol:
            break
        d = -np.dot(B, g)
        alpha = armijo(f, x, d, g, c1, 1.0, rho)
        s = alpha * d
        x_new = x + s
        g_new = grad(x_new)
        y = g_new - g
        rho_bfgs = 1.0 / (np.dot(y, s)) if np.abs(np.dot(y, s)) > 1e-12 else 1.0
        I = np.eye(n)
        term = I - rho_bfgs * np.outer(s, y)
        B = np.dot(np.dot(term, B), term.T) + rho_bfgs * np.outer(s, s)
        x = x_new
        g = g_new
        history.append(x.copy())
        values.append(f(x))
        n_iter += 1
        if np.linalg.norm(s) < tol:
            break
    nfev = len(history)
    njev = n_iter
    nhev = 0
    return x, np.array(history), np.array(values), n_iter, nfev, njev, nhev

# 5. Нелдер-Мид
def nelder_mead_with_history(f, x0, max_iter=1000, tol=1e-6):
    history = [np.array(x0)]
    values = [f(x0)]
    nfev = 1
    def callback(xk):
        history.append(np.array(xk))
        values.append(f(xk))
        nonlocal nfev
        nfev += 1
    res = minimize(f, x0, method='Nelder-Mead',
                   options={'maxiter': max_iter, 'xatol': tol}, callback=callback)
    nit = res.nit
    success = res.success
    return res.x, np.array(history), np.array(values), nit, success, nfev

# 6. Начальные точки
start_points = {
    'Booth': [[0.1, 0.1], [5.0, 5.0], [2.0, 2.0]],
    'Rastrigin': [[0.1, 0.1], [1.55, 1.55], [1.0, 0.5]],
    'Weierstrass': [[0.2, 0.2], [1.2, 1.2], [-0.8, 0.5]]
}

functions = {
    'Booth': (booth, booth_grad, booth_hess),
    'Rastrigin': (rastrigin, rastrigin_grad, rastrigin_hess),
    'Weierstrass': (weierstrass, weierstrass_grad, weierstrass_hess)
}

# 7. Запуск всех экспериментов
all_results = []
for fname, (f, grad, hess) in functions.items():
    for idx, x0 in enumerate(start_points[fname]):
        x_n, hist_n, vals_n, it_n, nfev_n, njev_n, nhev_n = newton_method(f, grad, hess, x0)
        all_results.append({
            'Function': fname, 'Start': tuple(x0), 'Method': 'Newton',
            'Iterations': it_n, 'Converged': it_n < 999, 'Final_f': f(x_n),
            'f_calls': nfev_n, 'grad_calls': njev_n, 'hess_calls': nhev_n,
            'history': hist_n, 'values': vals_n
        })
        x_b, hist_b, vals_b, it_b, nfev_b, njev_b, nhev_b = bfgs_own(f, grad, x0)
        all_results.append({
            'Function': fname, 'Start': tuple(x0), 'Method': 'BFGS',
            'Iterations': it_b, 'Converged': it_b < 999, 'Final_f': f(x_b),
            'f_calls': nfev_b, 'grad_calls': njev_b, 'hess_calls': nhev_b,
            'history': hist_b, 'values': vals_b
        })
        x_nm, hist_nm, vals_nm, it_nm, succ_nm, nfev_nm = nelder_mead_with_history(f, x0)
        all_results.append({
            'Function': fname, 'Start': tuple(x0), 'Method': 'Nelder-Mead',
            'Iterations': it_nm, 'Converged': succ_nm, 'Final_f': f(x_nm),
            'f_calls': nfev_nm, 'grad_calls': 0, 'hess_calls': 0,
            'history': hist_nm, 'values': vals_nm
        })

# 8. Вычисление глобального минимума для каждой функции
fstar = {}
for fname in functions.keys():
    best = min(r['Final_f'] for r in all_results if r['Function'] == fname)
    fstar[fname] = best
print("Приближённые глобальные минимумы:")
for fname, val in fstar.items():
    print(f"  {fname}: {val:.6e}")

# 9. Таблица результатов 
print("\n" + "="*130)
print(f"{'Function':<12} {'Start point':<20} {'Method':<12} {'Iter':<6} {'f_calls':<8} {'grad':<8} {'hess':<8} {'Converged':<8} {'Final f':<12}")
print("="*130)
for r in all_results:
    start_str = str(r['Start']).replace(' ', '')
    print(f"{r['Function']:<12} {start_str:<20} {r['Method']:<12} {r['Iterations']:<6} {r['f_calls']:<8} {r['grad_calls']:<8} {r['hess_calls']:<8} {int(r['Converged']):<8} {r['Final_f']:.4e}")
print("="*130)

# 10. Построение графиков
for fname in functions.keys():
    f, _, _ = functions[fname]
    for x0 in start_points[fname]:
        histories = {}
        values_dict = {}
        start_tuple = tuple(x0)
        for r in all_results:
            if r['Function'] == fname and r['Start'] == start_tuple:
                histories[r['Method']] = r['history']
                values_dict[r['Method']] = r['values']
        if not histories:
            continue

        if fname == 'Weierstrass':
            margin = 0.8
            bounds = [x0[0]-margin, x0[0]+margin, x0[1]-margin, x0[1]+margin]
        else:
            all_x, all_y = [], []
            for method in ['Newton', 'BFGS', 'Nelder-Mead']:
                if method in histories and histories[method] is not None and len(histories[method]) > 0:
                    all_x.extend(histories[method][:,0])
                    all_y.extend(histories[method][:,1])
            if all_x:
                min_x, max_x = min(all_x), max(all_x)
                min_y, max_y = min(all_y), max(all_y)
                dx = max(0.5, (max_x - min_x) * 0.2)
                dy = max(0.5, (max_y - min_y) * 0.2)
                bounds = [min_x - dx, max_x + dx, min_y - dy, max_y + dy]
            else:
                bounds = [-2, 6, -2, 6] if fname == 'Booth' else [-2, 3, -2, 3]

        x_grid = np.linspace(bounds[0], bounds[1], 300)
        y_grid = np.linspace(bounds[2], bounds[3], 300)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = np.array([f([xi, yi]) for xi, yi in zip(X.ravel(), Y.ravel())]).reshape(X.shape)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        methods_order = ['Newton', 'BFGS', 'Nelder-Mead']
        for idx, method in enumerate(methods_order):
            ax = axes[idx]
            ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6, linewidths=0.8)
            if method in histories and histories[method] is not None and len(histories[method]) > 1:
                hist = histories[method]
                ax.plot(hist[:,0], hist[:,1], 'r-', linewidth=1.5)
                ax.scatter(hist[0,0], hist[0,1], color='blue', s=40, label='start')
                ax.scatter(hist[-1,0], hist[-1,1], color='red', s=40, label='end')
            else:
                ax.scatter(x0[0], x0[1], color='blue', s=40, label='start')
            ax.set_xlim(bounds[0], bounds[1])
            ax.set_ylim(bounds[2], bounds[3])
            ax.set_title(method)
            ax.legend()
        plt.suptitle(f'{fname} from {x0} – trajectories')
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(10, 6))
        f_star = fstar[fname]
        for method in methods_order:
            vals = values_dict.get(method)
            if vals is not None and len(vals) > 0:
                diff = np.abs(np.array(vals) - f_star) + 1e-16
                plt.semilogy(range(len(diff)), diff, marker='.', label=method)
        plt.xlabel('Iteration')
        plt.ylabel('|f(x) - f*|')
        plt.title(f'Convergence on {fname} from {x0} (f* = {f_star:.3e})')
        plt.legend()
        plt.grid(True)
        plt.show()

# Вывод
- На выпуклых функциях Ньютон и BFGS работают отлично, Нелдер–Мид медленный.  
- На невыпуклых функциях с множеством локальных минимумов, ни один метод не гарантирует глобальный минимум – всё зависит от начальной точки, как и в первой лабораторной, если выбирать локальный минимум, методы будут идти к нему.  
- Модификация метода Ньютона (замена направления на антиградиент) предотвращает расходимость, но превращает его в градиентный спуск, который сходится к ближайшему локальному минимуму.  
- BFGS – лучший метод из 3 выбранных, работает не сильно дольше Ньютона, но зато не расходится, работает лучше, чем модификация Ньютона, превращающая его в град.спуск.
- На фрактальной функции метод Ньютона разошёлся, метод Нелдера-Мида не справился, а BFGS справился только один раз - повезло с точкой.
- Метод Нелдера–Мида стоит использовать только когда производные сложные и громоздкие или когда вычисление функции стоит довольно дорого, и за это увеличивается время работы.
- Метод Ньютона лучше на выпуклых функциях (чтобы не столкнуться с отрицательно определённым гессианом)
- BFGS можно использовать, когда гессиан положительно не определён или довольно громоздкий, и когда вычисление функции стоит дорого